# LCT — RF-DETR Medium @576

Архитектурный масштабный эксперимент после сильного результата RF-DETR Small.

**Input:** только `grigoriyyy/lct-base-combined-dataset`.

Используем официальный pretrained `RFDETRMedium` на его native `576×576`.
Ставим **6 эпох** и checkpoint каждую эпоху, чтобы не повторить 12-часовой timeout
Small run. Цель этой ночи — получить честную trajectory + лучший checkpoint,
а не любой ценой закончить десятки эпох.


In [1]:
!nvidia-smi
!pip install -q "rfdetr[train]==1.10.1" pyyaml pillow

import os
import json
import time
import random
import shutil
from pathlib import Path

import numpy as np
import torch
import yaml
import rfdetr
from PIL import Image

print("rfdetr:", getattr(rfdetr, "__version__", "unknown"))
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

from rfdetr import RFDETRMedium


Thu Sep 24 01:55:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
SEED = 42
RESOLUTION = 576
EPOCHS = 6
OUTPUT_DIR = Path("/kaggle/working/rfdetr_medium_576")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


In [3]:
# Locate frozen construction_v1
INPUT = Path("/kaggle/input")
yamls = list(INPUT.rglob("data.yaml"))

print("Found data.yaml files:")
for y in yamls:
    print(" ", y)

assert yamls, "No data.yaml found under /kaggle/input"

candidates = [y for y in yamls if "construction_v1" in y.as_posix()]
DATA_YAML = candidates[0] if candidates else yamls[0]
DATA_ROOT = DATA_YAML.parent

with open(DATA_YAML, "r", encoding="utf-8") as f:
    original_data = yaml.safe_load(f)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
expected_counts = {"train": 19391, "val": 4609, "test": 1102}

for split in ("train", "val", "test"):
    p = DATA_ROOT / original_data[split]
    n = sum(1 for x in p.rglob("*") if x.is_file() and x.suffix.lower() in IMAGE_EXTS)
    print(f"{split}: {n}")
    assert n == expected_counts[split], (split, n, expected_counts[split])

names_dict = original_data["names"]
assert len(names_dict) == 13
CLASS_NAMES = [names_dict[i] if i in names_dict else names_dict[str(i)] for i in range(13)]

print("DATA_ROOT =", DATA_ROOT)
print("Classes:", CLASS_NAMES)


Found data.yaml files:
  /kaggle/input/datasets/grigoriyyy/lct-base-combined-dataset/construction_v1/data.yaml
train: 19391
val: 4609
test: 1102
DATA_ROOT = /kaggle/input/datasets/grigoriyyy/lct-base-combined-dataset/construction_v1
Classes: ['excavator', 'dump_truck', 'truck', 'loader', 'bulldozer', 'motor_grader', 'roller', 'concrete_mixer', 'telehandler', 'piling_machine', 'crane_manipulator', 'mobile_crane', 'tower_crane']


In [4]:
# Build RF-DETR-compatible YOLO runtime view.
# Images are symlinked; labels are copied and exact duplicate rows are removed.
RF_DATA = Path("/kaggle/working/rfdetr_dataset")

if RF_DATA.exists():
    shutil.rmtree(RF_DATA)

for split in ("train", "valid"):
    (RF_DATA / split / "images").mkdir(parents=True, exist_ok=True)
    (RF_DATA / split / "labels").mkdir(parents=True, exist_ok=True)

def build_split(source_split: str, target_split: str):
    src_images = DATA_ROOT / "images" / source_split
    src_labels = DATA_ROOT / "labels" / source_split

    image_count = 0
    duplicate_lines_removed = 0
    seen_names = set()

    images = sorted(
        p for p in src_images.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

    for img in images:
        rel = img.relative_to(src_images)
        flat_stem = "__".join(list(rel.parts[:-1]) + [img.stem])
        target_name = flat_stem + img.suffix.lower()

        assert target_name not in seen_names, f"Collision: {target_name}"
        seen_names.add(target_name)

        os.symlink(img, RF_DATA / target_split / "images" / target_name)
        image_count += 1

        src_label = src_labels / rel.with_suffix(".txt")
        dst_label = RF_DATA / target_split / "labels" / (flat_stem + ".txt")

        if not src_label.exists():
            dst_label.write_text("", encoding="utf-8")
            continue

        lines = [x.strip() for x in src_label.read_text(encoding="utf-8").splitlines() if x.strip()]
        unique = list(dict.fromkeys(lines))
        duplicate_lines_removed += len(lines) - len(unique)
        dst_label.write_text(("\n".join(unique) + "\n") if unique else "", encoding="utf-8")

    return {"images": image_count, "duplicate_lines_removed": duplicate_lines_removed}

train_build = build_split("train", "train")
val_build = build_split("val", "valid")

rf_yaml = {
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    "nc": len(CLASS_NAMES),
    "train": "train/images",
    "val": "valid/images",
}
(RF_DATA / "data.yaml").write_text(
    yaml.safe_dump(rf_yaml, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

assert train_build["images"] == 19391
assert val_build["images"] == 4609

print("RF-DETR runtime view ready:")
print(" train:", train_build)
print(" valid:", val_build)
print(" RF_DATA:", RF_DATA)


RF-DETR runtime view ready:
 train: {'images': 19391, 'duplicate_lines_removed': 1}
 valid: {'images': 4609, 'duplicate_lines_removed': 0}
 RF_DATA: /kaggle/working/rfdetr_dataset


In [5]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model = RFDETRMedium()

t0 = time.perf_counter()

model.train(
    dataset_dir=str(RF_DATA),
    dataset_file="yolo",
    epochs=EPOCHS,
    resolution=RESOLUTION,
    batch_size="auto",
    grad_accum_steps=1,
    num_workers=4,
    output_dir=str(OUTPUT_DIR),
    seed=SEED,

    use_ema=True,
    eval_base_model=False,
    checkpoint_interval=1,

    early_stopping=False,
    skip_best_epochs=1,
    run_test=False,

    tensorboard=True,
    wandb=False,
    progress_bar="tqdm",
)

print("Training hours:", (time.perf_counter() - t0) / 3600)


[2026-09-24 01:57:48] [INFO] rf-detr - Downloading pretrained weights for /root/.roboflow/models/rf-detr-medium.pth


/root/.roboflow/models/rf-detr-medium.pth:   0%|          | 0.00/386M [00:00<?, ?iB/s]

[2026-09-24 01:58:03] [INFO] rf-detr - MD5 validation successful for /root/.roboflow/models/rf-detr-medium.pth


[2026-09-24 01:58:03] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-24 01:58:03] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-09-24 01:58:05] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-medium.pth already exists with correct MD5 hash.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

[2026-09-24 10:44:34] [INFO] rf-detr - Best EMA metric improved to 0.6713 (epoch 5)


`Trainer.fit` stopped: `max_epochs=6` reached.


[2026-09-24 10:44:39] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.0000, ema=0.6713)
Training hours: 8.775709656170834


In [6]:
print("\nOutput files:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(OUTPUT_DIR))

print("\nIMPORTANT: keep the best checkpoint and last.ckpt separately.")
print("No long post-training evaluation is run here: tonight's priority is training.")



Output files:
  checkpoint_0.ckpt
  checkpoint_1.ckpt
  checkpoint_2.ckpt
  checkpoint_3.ckpt
  checkpoint_4.ckpt
  checkpoint_5.ckpt
  checkpoint_best_ema.pth
  checkpoint_best_total.pth
  events.out.tfevents.1790215118.1ff61c59c460.23.0
  hparams.yaml
  last_ema.pth
  metrics.csv
  training_config.json

IMPORTANT: keep the best checkpoint and last.ckpt separately.
No long post-training evaluation is run here: tonight's priority is training.


Сохрани `checkpoint_best_total.pth`, `checkpoint_best_ema.pth`, `last.ckpt`
и `metrics.csv`. Organizer100 и per-source eval сделаем отдельно.
